In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [6]:
import torch
import torch.nn as nn

class Pairwise(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.n = n_features
        self.weight = nn.Parameter(torch.zeros(n_features, n_features))
        self.bias   = nn.Parameter(torch.ones(n_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:        
        # xw = torch.sigmoid(2 * (torch.einsum("bj,ij->bij", x, self.weight) + self.bias)) # (batch, n, n)
        # return xw.sum(dim=2) / self.n_2 * x
        xw = torch.einsum("bj,ij->bij", x, self.weight) # (batch, n, n)
        return (xw.sum(dim=2) + self.bias) * x

In [7]:
class DiamondNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = Pairwise(input_dim)
        self.fc2 = Pairwise(input_dim * 2)
        self.fc3 = Pairwise(input_dim * 4)
        self.fc4 = Pairwise(input_dim * 8)
        self.fc5 = nn.Linear(input_dim * 16, 2)

        for layer in [self.fc5]:
            nn.init.constant_(layer.weight, 0.0)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        new_x = self.fc1(x)
        x = torch.cat((x, new_x), dim=1)

        new_x = self.fc2(x)
        x = torch.cat((x, new_x), dim=1)

        new_x = self.fc3(x)
        x = torch.cat((x, new_x), dim=1)

        new_x = self.fc4(x)
        x = torch.cat((x, new_x), dim=1)

        x = self.fc5(x)
        return x

print(X.shape[1])

7


In [8]:
model = torch.compile(DiamondNN(input_dim=X.shape[1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")


Total trainable parameters: 4496
Epoch 1/10000, Train Loss: 0.6931, Test Loss: 0.7631, Test Accuracy: 0.6086
Epoch 2/10000, Train Loss: 0.7656, Test Loss: 0.6815, Test Accuracy: 0.6504
Epoch 3/10000, Train Loss: 0.6805, Test Loss: 0.4803, Test Accuracy: 0.7636
Epoch 4/10000, Train Loss: 0.4792, Test Loss: 0.4133, Test Accuracy: 0.8575
Epoch 5/10000, Train Loss: 0.4147, Test Loss: 0.3914, Test Accuracy: 0.8621
Epoch 6/10000, Train Loss: 0.3917, Test Loss: 0.3811, Test Accuracy: 0.8542
Epoch 7/10000, Train Loss: 0.3821, Test Loss: 0.3687, Test Accuracy: 0.8583
Epoch 8/10000, Train Loss: 0.3722, Test Loss: 0.3676, Test Accuracy: 0.8669
Epoch 9/10000, Train Loss: 0.3719, Test Loss: 0.3594, Test Accuracy: 0.8673
Epoch 10/10000, Train Loss: 0.3641, Test Loss: 0.3553, Test Accuracy: 0.8656
Epoch 11/10000, Train Loss: 0.3601, Test Loss: 0.3513, Test Accuracy: 0.8655
Epoch 12/10000, Train Loss: 0.3559, Test Loss: 0.3462, Test Accuracy: 0.8709
Epoch 13/10000, Train Loss: 0.3518, Test Loss: 0.343

KeyboardInterrupt: 

In [4]:
model = torch.compile(DiamondNN(input_dim=X.shape[1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")


Total trainable parameters: 8556
Epoch 1/10000, Train Loss: 0.6931, Test Loss: 0.7631, Test Accuracy: 0.6086
Epoch 2/10000, Train Loss: 0.7656, Test Loss: 0.6941, Test Accuracy: 0.6359
Epoch 3/10000, Train Loss: 0.6939, Test Loss: 0.5343, Test Accuracy: 0.7289
Epoch 4/10000, Train Loss: 0.5307, Test Loss: 0.4952, Test Accuracy: 0.8120
Epoch 5/10000, Train Loss: 0.4925, Test Loss: 0.4833, Test Accuracy: 0.8138
Epoch 6/10000, Train Loss: 0.4814, Test Loss: 0.4709, Test Accuracy: 0.8059
Epoch 7/10000, Train Loss: 0.4695, Test Loss: 0.4599, Test Accuracy: 0.8090
Epoch 8/10000, Train Loss: 0.4598, Test Loss: 0.4449, Test Accuracy: 0.8121
Epoch 9/10000, Train Loss: 0.4457, Test Loss: 0.4324, Test Accuracy: 0.8173
Epoch 10/10000, Train Loss: 0.4338, Test Loss: 0.4216, Test Accuracy: 0.8204
Epoch 11/10000, Train Loss: 0.4227, Test Loss: 0.4144, Test Accuracy: 0.8214
Epoch 12/10000, Train Loss: 0.4153, Test Loss: 0.4096, Test Accuracy: 0.8215
Epoch 13/10000, Train Loss: 0.4107, Test Loss: 0.404

KeyboardInterrupt: 

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = sns.load_dataset('diamonds')
df = df[['carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'cut']]

df['cut'] = (df['cut'] == 'Ideal').astype(int)

X = df.drop('cut', axis=1)
y = df['cut']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

class DiamondNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, input_dim)
        self.fc2 = nn.Linear(input_dim * 2, input_dim * 2)
        self.fc3 = nn.Linear(input_dim * 4, input_dim * 4)
        self.fc4 = nn.Linear(input_dim * 8, input_dim * 8)
        self.fc5 = nn.Linear(input_dim * 16, 2)

        for layer in [self.fc1, self.fc2, self.fc3, self.fc4, self.fc5]:
            nn.init.constant_(layer.weight, 0.0)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = x * 2
        new_x = torch.sigmoid(self.fc1(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc2(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc3(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = x * 2
        new_x = torch.sigmoid(self.fc4(x)) * x
        x = torch.cat((x, new_x), dim=1)

        x = self.fc5(x)
        return x

print(X.shape[1])

In [ ]:
model = torch.compile(DiamondNN(input_dim=X.shape[1]))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Rprop(model.parameters(), lr=0.01)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

for name, param in model.named_parameters():
    print(name, param)
    
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

In [ ]:
for name, param in model.named_parameters():
    print(name, param)

In [ ]:
model = DiamondNN(input_dim=X.shape[1])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {count_parameters(model)}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 10000

best_test_loss = float('inf')
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    train_outputs = model(X_train_tensor)
    train_loss = criterion(train_outputs, y_train_tensor)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_loss = criterion(test_outputs, y_test_tensor)
        test_predictions = torch.argmax(test_outputs, dim=1)
        test_accuracy = accuracy_score(y_test_tensor, test_predictions)

    # Track lowest test loss
    if test_loss.item() < best_test_loss:
        best_test_loss = test_loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss.item():.4f}, Test Loss: {test_loss.item():.4f}, Test Accuracy: {test_accuracy:.4f}")

print(f"\nLowest Test Loss: {best_test_loss:.4f}")

In [ ]:
import torch
import numpy as np

np.set_printoptions(suppress=True, linewidth=200, precision=8)

for name, param in model.named_parameters():
    print(name)
    print(param.detach().cpu().numpy())
